In [ ]:

import pyodbc
import ollama


# ==================================================
# 1. SQL SERVER CONNECTION
# ==================================================

connection = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=DESKTOP-NB27H6F;"
    "DATABASE=orderssystem;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

cursor = connection.cursor()


# ==================================================
# 2. FUNCTION TO FETCH SQL DATA
# ==================================================

def get_data(query):

    cursor.execute(query)

    columns = [column[0] for column in cursor.description]

    rows = cursor.fetchall()

    result = []

    for row in rows:

        record = {}

        for i, column in enumerate(columns):
            record[column] = row[i]

        result.append(record)

    return result


# ==================================================
# 3. FORMAT SQL DATA
# ==================================================

def format_data(data):

    if not data:
        return "No records found."

    lines = []

    for record in data:

        row_data = []

        for key, value in record.items():

            row_data.append(f"{key}: {value}")

        lines.append(" | ".join(row_data))

    return "\n".join(lines)


# ==================================================
# 4. OVERALL PAYMENT SUMMARY
# ==================================================

def get_payment_summary():

    query = """
    SELECT

        COUNT(*) AS totalpaymentrecords,

        COUNT(DISTINCT expenseid) AS expenseswithpayments,

        ISNULL(SUM(paidamount), 0) AS totalrecordedpayments,

        ISNULL(SUM(
            CASE
                WHEN status = 'Paid'
                THEN paidamount
                ELSE 0
            END
        ), 0) AS totalpaid,

        ISNULL(SUM(
            CASE
                WHEN status = 'Pending'
                THEN paidamount
                ELSE 0
            END
        ), 0) AS totalpending,

        ISNULL(SUM(
            CASE
                WHEN status = 'Failed'
                THEN paidamount
                ELSE 0
            END
        ), 0) AS totalfailed

    FROM payment;
    """

    return get_data(query)


# ==================================================
# 5. PAYMENT STATUS ANALYSIS
# ==================================================

def get_payment_status_analysis():

    query = """
    SELECT

        status,

        COUNT(*) AS paymentcount,

        SUM(paidamount) AS totalamount,

        AVG(paidamount) AS averagepayment,

        MIN(paidamount) AS minimumpayment,

        MAX(paidamount) AS maximumpayment

    FROM payment

    GROUP BY status

    ORDER BY totalamount DESC;
    """

    return get_data(query)


# ==================================================
# 6. EXPENSE-WISE PAYMENT ANALYSIS
# ==================================================

def get_expense_payment_analysis():

    query = """
    WITH PaymentTotals AS
    (
        SELECT

            expenseid,

            SUM(CASE
                WHEN status = 'Paid'
                THEN paidamount
                ELSE 0
            END) AS totalpaid,

            SUM(CASE
                WHEN status = 'Pending'
                THEN paidamount
                ELSE 0
            END) AS pendingamount,

            SUM(CASE
                WHEN status = 'Failed'
                THEN paidamount
                ELSE 0
            END) AS failedamount

        FROM payment

        GROUP BY expenseid
    )

    SELECT TOP 100

        e.expenseid,

        e.expensedate,

        ec.categoryname,

        v.vendorname,

        e.description,

        e.amount AS expenseamount,

        ISNULL(p.totalpaid, 0) AS totalpaid,

        ISNULL(p.pendingamount, 0) AS pendingamount,

        ISNULL(p.failedamount, 0) AS failedamount,

        e.amount - ISNULL(p.totalpaid, 0)
            AS outstandingamount,

        CASE

            WHEN ISNULL(p.totalpaid, 0) > e.amount
                THEN 'Overpaid'

            WHEN ISNULL(p.totalpaid, 0) = e.amount
                THEN 'Fully Paid'

            WHEN ISNULL(p.totalpaid, 0) > 0
                THEN 'Partially Paid'

            ELSE 'Unpaid'

        END AS paymentposition

    FROM expense e

    INNER JOIN expensecategory ec
        ON e.categoryid = ec.categoryid

    LEFT JOIN vendor v
        ON e.vendorid = v.vendorid

    LEFT JOIN PaymentTotals p
        ON e.expenseid = p.expenseid

    ORDER BY e.expensedate DESC;
    """

    return get_data(query)


# ==================================================
# 7. VENDOR-WISE PAYMENT ANALYSIS
# ==================================================

def get_vendor_payment_analysis():

    query = """
    WITH PaymentTotals AS
    (
        SELECT

            expenseid,

            SUM(CASE
                WHEN status = 'Paid'
                THEN paidamount
                ELSE 0
            END) AS totalpaid,

            SUM(CASE
                WHEN status = 'Pending'
                THEN paidamount
                ELSE 0
            END) AS pendingamount,

            SUM(CASE
                WHEN status = 'Failed'
                THEN paidamount
                ELSE 0
            END) AS failedamount

        FROM payment

        GROUP BY expenseid
    )

    SELECT

        COALESCE(v.vendorname, 'Unassigned / No vendor')
            AS vendorname,

        COUNT(e.expenseid) AS expensecount,

        ISNULL(SUM(e.amount), 0) AS totalexpense,

        ISNULL(SUM(p.totalpaid), 0) AS totalpaid,

        ISNULL(SUM(p.pendingamount), 0) AS pendingpayment,

        ISNULL(SUM(p.failedamount), 0) AS failedpayment,

        ISNULL(
            SUM(e.amount - ISNULL(p.totalpaid, 0)), 0
        ) AS outstandingamount

    FROM expense e

    LEFT JOIN vendor v
        ON e.vendorid = v.vendorid

    LEFT JOIN PaymentTotals p
        ON e.expenseid = p.expenseid

    GROUP BY v.vendorname

    ORDER BY totalexpense DESC;
    """

    return get_data(query)


# ==================================================
# 8. CATEGORY-WISE PAYMENT ANALYSIS
# ==================================================

def get_category_payment_analysis():

    query = """
    WITH PaymentTotals AS
    (
        SELECT

            expenseid,

            SUM(CASE
                WHEN status = 'Paid'
                THEN paidamount
                ELSE 0
            END) AS totalpaid,

            SUM(CASE
                WHEN status = 'Pending'
                THEN paidamount
                ELSE 0
            END) AS pendingamount,

            SUM(CASE
                WHEN status = 'Failed'
                THEN paidamount
                ELSE 0
            END) AS failedamount

        FROM payment

        GROUP BY expenseid
    )

    SELECT

        ec.categoryname,

        COUNT(e.expenseid) AS expensecount,

        SUM(e.amount) AS totalexpense,

        SUM(ISNULL(p.totalpaid, 0)) AS totalpaid,

        SUM(ISNULL(p.pendingamount, 0)) AS pendingpayment,

        SUM(ISNULL(p.failedamount, 0)) AS failedpayment,

        SUM(
            e.amount - ISNULL(p.totalpaid, 0)
        ) AS outstandingamount

    FROM expense e

    INNER JOIN expensecategory ec
        ON e.categoryid = ec.categoryid

    LEFT JOIN PaymentTotals p
        ON e.expenseid = p.expenseid

    GROUP BY ec.categoryname

    ORDER BY totalexpense DESC;
    """

    return get_data(query)


# ==================================================
# 9. MONTHLY PAYMENT ANALYSIS
# ==================================================

def get_monthly_payment_analysis():

    query = """
    SELECT

        YEAR(paymentdate) AS paymentyear,

        MONTH(paymentdate) AS paymentmonth,

        status,

        COUNT(*) AS paymentcount,

        SUM(paidamount) AS totalamount,

        AVG(paidamount) AS averagepayment

    FROM payment

    GROUP BY

        YEAR(paymentdate),

        MONTH(paymentdate),

        status

    ORDER BY

        paymentyear,
        paymentmonth,
        status;
    """

    return get_data(query)


# ==================================================
# 10. PAYMENT METHOD ANALYSIS
# ==================================================

def get_payment_method_analysis():

    query = """
    SELECT

        COALESCE(paymentmethod, 'Not specified')
            AS paymentmethod,

        status,

        COUNT(*) AS paymentcount,

        SUM(paidamount) AS totalamount

    FROM payment

    GROUP BY
        paymentmethod,
        status

    ORDER BY totalamount DESC;
    """

    return get_data(query)


# ==================================================
# 11. UNPAID AND PARTIALLY PAID EXPENSES
# ==================================================

def get_unpaid_expenses():

    query = """
    WITH PaymentTotals AS
    (
        SELECT

            expenseid,

            SUM(CASE
                WHEN status = 'Paid'
                THEN paidamount
                ELSE 0
            END) AS totalpaid

        FROM payment

        GROUP BY expenseid
    )

    SELECT

        e.expenseid,

        e.expensedate,

        ec.categoryname,

        v.vendorname,

        e.description,

        e.amount AS expenseamount,

        ISNULL(p.totalpaid, 0) AS totalpaid,

        e.amount - ISNULL(p.totalpaid, 0)
            AS outstandingamount,

        DATEDIFF(
            DAY,
            e.expensedate,
            GETDATE()
        ) AS expenseageindays

    FROM expense e

    LEFT JOIN PaymentTotals p
        ON e.expenseid = p.expenseid

    INNER JOIN expensecategory ec
        ON e.categoryid = ec.categoryid

    LEFT JOIN vendor v
        ON e.vendorid = v.vendorid

    WHERE
        e.amount - ISNULL(p.totalpaid, 0) > 0

    ORDER BY outstandingamount DESC;
    """

    return get_data(query)


# ==================================================
# 12. TOP 10 PAYMENT TRANSACTIONS
# ==================================================

def get_top_payments():

    query = """
    SELECT TOP 10

        p.paymentid,

        p.expenseid,

        p.paymentdate,

        v.vendorname,

        ec.categoryname,

        p.paymentmethod,

        p.status,

        p.paidamount

    FROM payment p

    INNER JOIN expense e
        ON p.expenseid = e.expenseid

    INNER JOIN expensecategory ec
        ON e.categoryid = ec.categoryid

    LEFT JOIN vendor v
        ON e.vendorid = v.vendorid

    ORDER BY p.paidamount DESC;
    """

    return get_data(query)


# ==================================================
# 13. LOAD DATABASE INFORMATION
# ==================================================

def load_payment_data():

    print("\nLoading payment information from SQL Server...")

    payment_summary = get_payment_summary()

    status_analysis = get_payment_status_analysis()

    expense_analysis = get_expense_payment_analysis()

    vendor_analysis = get_vendor_payment_analysis()

    category_analysis = get_category_payment_analysis()

    monthly_analysis = get_monthly_payment_analysis()

    method_analysis = get_payment_method_analysis()

    unpaid_analysis = get_unpaid_expenses()

    top_payments = get_top_payments()

    all_data = f"""

    OVERALL PAYMENT SUMMARY
    -----------------------
    {format_data(payment_summary)}


    PAYMENT STATUS ANALYSIS
    -----------------------
    {format_data(status_analysis)}


    EXPENSE-WISE PAYMENT ANALYSIS
    -----------------------------
    {format_data(expense_analysis)}


    VENDOR-WISE PAYMENT ANALYSIS
    ----------------------------
    {format_data(vendor_analysis)}


    CATEGORY-WISE PAYMENT ANALYSIS
    ------------------------------
    {format_data(category_analysis)}


    MONTHLY PAYMENT ANALYSIS
    ------------------------
    {format_data(monthly_analysis)}


    PAYMENT METHOD ANALYSIS
    -----------------------
    {format_data(method_analysis)}


    UNPAID AND PARTIALLY PAID EXPENSES
    ---------------------------------
    {format_data(unpaid_analysis)}


    TOP 10 PAYMENT TRANSACTIONS
    ---------------------------
    {format_data(top_payments)}

    """

    return all_data


# ==================================================
# 14. PAYMENT ANALYSIS AGENT
# ==================================================

def payment_analysis_agent(user_question):

    all_data = load_payment_data()

    system_prompt = """
    You are an expert Payment Analysis Agent.

    You analyse business payments and expense settlements
    using SQL Server business data.

    YOUR RESPONSIBILITIES:

    1. Total paid payments.
    2. Pending payment analysis.
    3. Failed payment analysis.
    4. Vendor-wise payment analysis.
    5. Category-wise payment analysis.
    6. Expense-wise payment status.
    7. Outstanding expense balances.
    8. Partial payment identification.
    9. Overpayment identification.
    10. Monthly payment trends.
    11. Payment method analysis.
    12. Identification of old unpaid expenses.
    13. Payment management recommendations.

    IMPORTANT RULES:

    - Use only the supplied database information.
    - Never invent payment records or financial figures.
    - Only status 'Paid' counts as a successful payment.
    - Status 'Pending' is not a completed payment.
    - Status 'Failed' is not a completed payment.
    - Never add payment amounts to expense amounts.
    - Outstanding = expense amount minus successful payments.
    - Identify overpayments separately.
    - Pending and failed amounts must be reported separately.
    - Do not assume an expense is overdue because it is old.
    - The database does not contain a payment due date.
    - Expense age is not the same as payment delay.
    - If information is unavailable, say so.
    - Explain calculations clearly.
    - Show financial figures with two decimal places.
    - Give practical business recommendations.
    - Use simple professional business language.
    """

    user_prompt = f"""
    PAYMENT DATABASE INFORMATION:

    {all_data}

    USER QUESTION:

    {user_question}

    Analyse the supplied data and answer the question.
    """

    response = ollama.chat(
        model="llama3.2",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    return response["message"]["content"]


# ==================================================
# 15. INTERACTIVE USER INTERFACE
# ==================================================

print("\n======================================")
print("       PAYMENT ANALYSIS AGENT")
print("======================================")

print("Database: orderssystem")
print("AI Model: llama3.2")

print("\nType 'exit' to close the agent.")

while True:

    question = input("\nAsk your payment analysis question: ")

    if question.lower().strip() == "exit":
        break

    if not question.strip():
        print("Please enter a question.")
        continue

    try:

        answer = payment_analysis_agent(question)

        print("\nAI PAYMENT ANALYSIS")
        print("--------------------------------------")

        print(answer)

        print("--------------------------------------")

    except Exception as e:

        print("\nError:", e)


# ==================================================
# 16. CLOSE DATABASE CONNECTION
# ==================================================

cursor.close()

connection.close()

print("\nPayment Analysis Agent stopped.")



       PAYMENT ANALYSIS AGENT
Database: orderssystem
AI Model: llama3.2

Type 'exit' to close the agent.



Ask your payment analysis question:  show vendor wise payment no explanation



Loading payment information from SQL Server...

AI PAYMENT ANALYSIS
--------------------------------------
Based on the provided payment database information, I will show the vendor-wise payment analysis as requested:

Vendor-wise Payment Analysis:
-----------------------------

| Vendorname | Expensecount | Totalexpense | Totalpaid | Pendingpayment | Failedpayment | Outstandingamount |
| --- | --- | --- | --- | --- | --- | --- |
| ABC Properties | 2 | 24000.00 | 12000.00 | 6000.00 | 0.00 | 12000.00 |
| Electricity Provider | 2 | 4000.00 | 4000.00 | 0.00 | 0.00 | 0.00 |
| Travel Services | 1 | 2500.00 | 2500.00 | 0.00 | 0.00 | 0.00 |
| Internet Provider | 2 | 1998.00 | 999.00 | 0.00 | 0.00 | 999.00 |
| Office Supplies Store | 1 | 1500.00 | 1500.00 | 0.00 | 0.00 | 0.00 |
--------------------------------------



Ask your payment analysis question:  show all the outstanding expences



Loading payment information from SQL Server...
